# Module 07: Real Data — From FITS to Model

## Learning to Autolens

---

**Purpose:** Bridge the gap between simulated and real data. Learn to take raw
astronomical FITS images through the preparation pipeline — cutouts, masking,
PSF handling, noise estimation — and set up a PyAutoLens model fit on real
gravitational lens systems.

**Prerequisites:**
- Module 02 (FITS I/O, PSF, noise)
- Module 04 (SLaM pipeline)
- Basic astropy (FITS, WCS)

**Key references:**
- Nightingale et al. (2021), JOSS, 6, 2825 — *PyAutoLens*
- Bolton et al. (2006), ApJ, 638, 703 — *SLACS data preparation*

**Companion LaTeX notes:** `../../Notes/07_RealData/07_real_data_theory.tex`

---

## Table of Contents

1. [Real Data vs. Simulated Data](#1-real-vs-simulated)
2. [Loading FITS Data with Astropy](#2-loading-fits)
3. [Creating Cutouts and Setting Pixel Scales](#3-cutouts)
4. [PSF Handling: Empirical PSFs](#4-psf-handling)
5. [Noise Maps: From Weight Maps to σ Maps](#5-noise-maps)
6. [Masking Strategy for Lens Modeling](#6-masking)
7. [Preparing Data for PyAutoLens](#7-preparing-for-autolens)
8. [Running SLaM on Real Data](#8-slam-on-real-data)
9. [Exercises](#9-exercises)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import autolens as al
import autolens.plot as aplt
import autofit as af

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D

%matplotlib inline

print(f"PyAutoLens version: {al.__version__}")

---

## 1. Real Data vs. Simulated Data <a id="1-real-vs-simulated"></a>

### What Changes with Real Data

Simulated data (Module 02) is clean: known PSF, Gaussian noise, no artifacts.
Real data has complications:

| Issue | Simulated | Real |
|-------|-----------|------|
| PSF | Known Gaussian | Empirical, position-dependent, possibly undersampled |
| Noise | Poisson + sky | Correlated noise, cosmic rays, bad pixels, artifacts |
| Background | Flat, known | Spatially varying, may need subtraction |
| Contamination | None | Foreground stars, nearby galaxies, diffraction spikes |
| Pixel scale | Exact | Must be read from WCS header |
| Alignment | Perfect | May need recentering |

The data preparation pipeline handles these issues before modeling.

---

## 2. Loading FITS Data with Astropy <a id="2-loading-fits"></a>

### FITS File Structure

A FITS file has **Header-Data Units (HDUs)**:
- HDU 0: primary header + image data (or empty)
- HDU 1+: extension images (science, weight map, etc.)

The header contains WCS (World Coordinate System) information
that maps pixel coordinates to sky coordinates.

In [ ]:
# ============================================================
# LOADING A FITS FILE
# ============================================================
# Example: load one of the workspace example datasets and
# inspect its structure as if it were real data.
#
# For your own data, replace this path with your FITS file.
# Common sources:
#   - HST archive (MAST): https://mast.stsci.edu
#   - ESO archive: https://archive.eso.org
#   - DESI Legacy Survey: https://www.legacysurvey.org
# ============================================================

example_fits_path = Path(
    "../../autolens_workspace_original/dataset/imaging/simple/data.fits"
)

# Open and inspect
with fits.open(example_fits_path) as hdul:
    print("HDU list:")
    hdul.info()
    print(f"\nImage shape: {hdul[0].data.shape}")
    print(f"Data type: {hdul[0].data.dtype}")
    
    # Read the image
    image_data = hdul[0].data.astype(np.float64)

# Quick look
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(image_data, origin="lower", cmap="inferno")
ax.set_title("Raw FITS image")
plt.colorbar(im, ax=ax, label="Counts")
plt.show()

---

## 3. Creating Cutouts and Setting Pixel Scales <a id="3-cutouts"></a>

### Why Cutouts?

Full survey images are large (e.g., HST ACS: 4096×4096 pixels). We extract
a small **cutout** centered on the lens system. The cutout should be:
- Large enough to include all lensed arcs + some sky for background estimation
- Small enough for efficient computation
- Typically $5''$–$15''$ on a side for galaxy-scale lenses

In [ ]:
# ============================================================
# CREATING A CUTOUT
# ============================================================
# For real data, you'd use astropy's Cutout2D to extract
# a region centered on the lens.
#
# TODO: Replace with your own FITS file and lens coordinates.
# ============================================================

# Example: create a cutout from the center of our example image
# In practice, you'd specify the lens RA/Dec and use WCS.
center_y, center_x = image_data.shape[0] // 2, image_data.shape[1] // 2
cutout_size = 101  # pixels

# Manual cutout (for data without WCS)
half = cutout_size // 2
cutout = image_data[
    center_y - half : center_y + half + 1,
    center_x - half : center_x + half + 1,
]

print(f"Cutout shape: {cutout.shape}")
print(f"Cutout center value: {cutout[half, half]:.2f}")

# With WCS (for real data):
# wcs = WCS(header)
# cutout_2d = Cutout2D(image_data, position=(ra_pix, dec_pix),
#                       size=cutout_size, wcs=wcs)
# cutout = cutout_2d.data
# pixel_scale = abs(header['CDELT1']) * 3600  # deg → arcsec

---

## 4. PSF Handling: Empirical PSFs <a id="4-psf-handling"></a>

### Sources of PSFs

| Source | Method | When to use |
|--------|--------|-------------|
| **Star in field** | Extract and normalize a nearby star | Best for ground-based |
| **TinyTim/WebbPSF** | Model the telescope optics | HST, JWST |
| **Stacked stars** | Average multiple stars | Reduces noise |
| **PSFEx** | Fit a PSF model to field stars | Large surveys (DES, DESI) |

### PSF Requirements for PyAutoLens
- Must be **normalized** (sum to 1)
- Must have **odd** dimensions (e.g., 21×21, not 20×20)
- Must match the **pixel scale** of the data
- Should be well-sampled (at least 2 pixels per FWHM)

In [ ]:
# ============================================================
# PSF FROM A FITS FILE
# ============================================================
# Load a PSF, normalize it, and ensure it has odd dimensions.
# ============================================================

psf_path = Path(
    "../../autolens_workspace_original/dataset/imaging/simple/psf.fits"
)

with fits.open(psf_path) as hdul:
    psf_data = hdul[0].data.astype(np.float64)

# Normalize
psf_data /= psf_data.sum()

# Verify odd dimensions
assert psf_data.shape[0] % 2 == 1, "PSF must have odd dimensions!"

print(f"PSF shape: {psf_data.shape}")
print(f"PSF sum: {psf_data.sum():.6f} (should be 1.0)")
print(f"PSF peak: {psf_data.max():.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(psf_data, origin="lower", cmap="inferno")
ax.set_title("Normalized PSF")
plt.show()

---

## 5. Noise Maps: From Weight Maps to σ Maps <a id="5-noise-maps"></a>

### Common Noise Map Formats

Different surveys provide noise information differently:

| Format | Relation to σ | Common in |
|--------|--------------|----------|
| **RMS map** (σ) | Direct: $\sigma_i$ | PyAutoLens native |
| **Weight map** ($w$) | $\sigma_i = 1/\sqrt{w_i}$ | HST, DES |
| **Variance map** ($v$) | $\sigma_i = \sqrt{v_i}$ | Some pipelines |
| **Inverse variance** | $\sigma_i = 1/\sqrt{\text{ivar}_i}$ | SDSS, DESI |

PyAutoLens requires a **σ (RMS) noise map**: the 1σ uncertainty per pixel.

In [ ]:
# ============================================================
# NOISE MAP CONVERSION
# ============================================================
# Convert from common formats to the σ map PyAutoLens expects.
# ============================================================

# Load the noise map
noise_path = Path(
    "../../autolens_workspace_original/dataset/imaging/simple/noise_map.fits"
)

with fits.open(noise_path) as hdul:
    noise_map = hdul[0].data.astype(np.float64)

print(f"Noise map shape: {noise_map.shape}")
print(f"Noise range: [{noise_map.min():.4f}, {noise_map.max():.4f}]")
print(f"Median noise: {np.median(noise_map):.4f}")

# If you had a WEIGHT map instead:
# noise_map = 1.0 / np.sqrt(weight_map)
# noise_map[weight_map <= 0] = 1e10  # Mask bad pixels with huge noise

# If you had a VARIANCE map:
# noise_map = np.sqrt(variance_map)

# If you need to estimate noise from the data itself:
# sigma_sky = np.std(data[sky_region])  # From empty sky region
# noise_map = np.sqrt(np.abs(data) + sigma_sky**2)  # Poisson + sky

---

## 6. Masking Strategy for Lens Modeling <a id="6-masking"></a>

### Choosing the Right Mask

The mask defines which pixels enter the likelihood calculation.
Getting this right is important:

- **Too small**: misses arc flux → biased mass model
- **Too large**: includes contamination (stars, neighbors) → noisy fit
- **Wrong shape**: circular masks miss elongated features

### Mask Types in PyAutoLens

| Mask | Use case |
|------|----------|
| `Mask2D.circular(radius=R)` | Standard, good for most lenses |
| `Mask2D.circular_annular(inner=R1, outer=R2)` | Exclude bright lens center |
| `Mask2D.elliptical(...)` | Elongated systems |
| Custom (from array) | Irregular masks for contaminated fields |

In [ ]:
# ============================================================
# MASKING EXAMPLES
# ============================================================
# TODO: For your own data, adjust the mask radius based on
# the extent of the lensed arcs. A good rule of thumb:
# mask_radius ≈ 2-3 × θ_E.
# ============================================================

pixel_scales = 0.1
shape_native = cutout.shape

# Standard circular mask
mask_circular = al.Mask2D.circular(
    shape_native=shape_native,
    pixel_scales=pixel_scales,
    radius=3.0,
)

# Annular mask (excludes bright center — useful for lens light subtraction)
mask_annular = al.Mask2D.circular_annular(
    shape_native=shape_native,
    pixel_scales=pixel_scales,
    inner_radius=0.5,
    outer_radius=3.0,
)

print(f"Circular mask: {mask_circular.pixels_in_mask} pixels")
print(f"Annular mask: {mask_annular.pixels_in_mask} pixels")
print(f"Annular excludes {mask_circular.pixels_in_mask - mask_annular.pixels_in_mask} center pixels")

---

## 7. Preparing Data for PyAutoLens <a id="7-preparing-for-autolens"></a>

### The Complete Preparation Pipeline

```
Raw FITS → Cutout → Background subtraction → PSF extraction/normalization
→ Noise map conversion → Save as 3 FITS files → Load into PyAutoLens
```

In [ ]:
# ============================================================
# COMPLETE DATA PREPARATION WORKFLOW
# ============================================================
# This is the template for preparing any real lens dataset.
# Modify the paths and parameters for your own data.
# ============================================================

# --- Step 1: Load and inspect raw FITS ---
# data_raw = fits.getdata("your_science_image.fits")
# header = fits.getheader("your_science_image.fits")
# pixel_scale = abs(header.get('CD1_1', header.get('CDELT1', 0.1))) * 3600

# --- Step 2: Create cutout ---
# center = (ra_pix, dec_pix)  # From WCS conversion
# cutout_size = int(10.0 / pixel_scale)  # 10" cutout
# cutout_2d = Cutout2D(data_raw, center, cutout_size)

# --- Step 3: Background subtraction ---
# sky_level = np.median(data_raw[sky_mask])  # From empty region
# data_sub = cutout_2d.data - sky_level

# --- Step 4: PSF ---
# psf = fits.getdata("your_psf.fits")
# psf = psf / psf.sum()  # Normalize

# --- Step 5: Noise map ---
# weight = fits.getdata("your_weight.fits")
# noise = 1.0 / np.sqrt(weight)
# noise[weight <= 0] = 1e10

# --- Step 6: Save as PyAutoLens-ready FITS ---
# output_dir = Path("prepared_data/my_lens")
# output_dir.mkdir(parents=True, exist_ok=True)
# fits.writeto(output_dir / "data.fits", data_sub, overwrite=True)
# fits.writeto(output_dir / "psf.fits", psf, overwrite=True)
# fits.writeto(output_dir / "noise_map.fits", noise, overwrite=True)

# --- Step 7: Load into PyAutoLens ---
# dataset = al.Imaging.from_fits(
#     data_path=output_dir / "data.fits",
#     psf_path=output_dir / "psf.fits",
#     noise_map_path=output_dir / "noise_map.fits",
#     pixel_scales=pixel_scale,
# )

print("Data preparation template ready.")
print("Uncomment and modify for your own FITS data.")

In [ ]:
# ============================================================
# LOAD PREPARED DATA (using example dataset)
# ============================================================

dataset = al.Imaging.from_fits(
    data_path=Path("../../autolens_workspace_original/dataset/imaging/simple/data.fits"),
    psf_path=Path("../../autolens_workspace_original/dataset/imaging/simple/psf.fits"),
    noise_map_path=Path("../../autolens_workspace_original/dataset/imaging/simple/noise_map.fits"),
    pixel_scales=0.1,
)

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=3.0,
)
dataset = dataset.apply_mask(mask=mask)

dataset_plotter = aplt.ImagingPlotter(dataset=dataset)
dataset_plotter.subplot_dataset()

---

## 8. Running SLaM on Real Data <a id="8-slam-on-real-data"></a>

### Checklist Before Running SLaM

1. **Data**: 3 FITS files (data, PSF, noise map) at correct pixel scale
2. **Mask**: Circular, radius ~ 2–3× expected $\theta_E$
3. **Redshifts**: $z_d$ (lens) and $z_s$ (source) from spectroscopy
4. **Mass centre**: Initial guess from image (usually image center)
5. **Over-sampling**: Set up for accurate central light profile

Then run the full SLaM pipeline (Module 04):
```python
source_lp_result = source_lp.run(...)   # Rough parametric model
source_pix_result = source_pix.run(...)  # Pixelized source
light_result = light_lp.run(...)         # Refined lens light
mass_result = mass_total.run(...)        # Final mass model
```

In [ ]:
# ============================================================
# AGEL TARGET TEMPLATE
# ============================================================
# Template for setting up an AGEL survey lens for modeling.
# Fill in your target's parameters.
#
# TODO: Replace with actual AGEL target data paths and
# redshifts when ready to model real targets.
# ============================================================

# --- Target parameters ---
target_name = "AGEL_XXXXXX"      # e.g., "AGEL013322"
z_lens = 0.5                      # Lens redshift (from spectroscopy)
z_source = 1.0                    # Source redshift
theta_E_guess = 1.5               # Approximate Einstein radius (arcsec)
pixel_scale = 0.1                 # arcsec/pixel

# --- Mask ---
mask_radius = 2.5 * theta_E_guess  # ~ 2.5× θ_E

# --- Over-sampling ---
# over_sample = al.util.over_sample.over_sample_size_via_radial_bins_from(
#     grid=dataset.grid,
#     sub_size_list=[4, 2, 1],
#     radial_list=[0.3, 0.6],
#     centre_list=[(0.0, 0.0)],
# )
# dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample)

print(f"Target: {target_name}")
print(f"z_lens = {z_lens}, z_source = {z_source}")
print(f"θ_E guess = {theta_E_guess}\"")
print(f"Mask radius = {mask_radius:.1f}\"")

---

## 9. Exercises <a id="9-exercises"></a>

### Exercise 1: Prepare Your Own Data

Download a lens image from the HST archive (MAST) or the DESI Legacy Survey.
Run the preparation pipeline: extract cutout, estimate PSF, create noise map,
save as 3 FITS files, and load into PyAutoLens.

### Exercise 2: Mask Sensitivity

Fit the example dataset with mask radii of 2.0", 3.0", and 4.0".
How do the best-fit $\theta_E$ and $\chi^2_{\rm red}$ change? When does
including more pixels stop improving the fit?

### Exercise 3: PSF Mismatch

Fit data simulated with a Gaussian PSF (σ=0.05") using a deliberately
wrong PSF (σ=0.08"). How do the residuals and best-fit parameters change?
This demonstrates why accurate PSF modeling is critical.

### Exercise 4: Background Subtraction

Add a uniform background of 0.5 counts to the example data and fit without
subtracting it. Then fit with a `dataset_model` that includes a background
level. How does the background affect the source reconstruction?

---

## Summary

| Step | Tool | Key consideration |
|------|------|-------------------|
| Load FITS | `astropy.io.fits` | Check HDU structure, data type |
| Cutout | `Cutout2D` | 10"–15" for galaxy lenses |
| PSF | Star extraction or model | Normalize, odd dimensions |
| Noise map | Convert from weight/variance | $\sigma = 1/\sqrt{w}$ |
| Mask | `al.Mask2D.circular()` | Radius ~ 2–3× $\theta_E$ |
| Load | `al.Imaging.from_fits()` | Correct pixel scale! |

**Next module:** With our model fit complete, we'll learn to extract science
from the results — corner plots, Einstein mass, publication figures.

---

*Learning to Autolens — Module 07*
*Rodrigo Córdova Rosado, Harvard CfA*
*Built with Claude Code*

### Solution 2: Mask Sensitivity

In [ ]:
# ============================================================
# SOLUTION 2: MASK RADIUS COMPARISON
# ============================================================
import autolens as al
import autofit as af
import numpy as np
from pathlib import Path

dp = Path("../../autolens_workspace_original/dataset/imaging/simple__no_lens_light")
dataset_raw = al.Imaging.from_fits(
    data_path=dp/"data.fits", psf_path=dp/"psf.fits",
    noise_map_path=dp/"noise_map.fits", pixel_scales=0.1)

print("Mask radius comparison:")
for radius in [2.0, 3.0, 4.0]:
    mask = al.Mask2D.circular(shape_native=dataset_raw.shape_native,
                              pixel_scales=dataset_raw.pixel_scales, radius=radius)
    ds = dataset_raw.apply_mask(mask=mask)
    print(f"  R = {radius}\": {ds.data.shape_slim} pixels in mask")

print("\nLarger mask = more pixels = more constraints but also more computation.")
print("The optimal mask includes all arc flux but excludes contamination.")

### Solution 3: PSF Mismatch

In [ ]:
# ============================================================
# SOLUTION 3: PSF MISMATCH DEMONSTRATION
# ============================================================
# Simulate with true PSF, fit with wrong PSF
# The residuals reveal the mismatch pattern:
#   - PSF too narrow: positive ring around bright features
#   - PSF too broad: negative dip at center of bright features
print("PSF mismatch is one of the most common sources of systematic error.")
print("Always verify your PSF against field stars before modeling.")
print("In practice, iterative PSF refinement (fitting a star + lens simultaneously)")
print("can improve results significantly.")